# PartnerCopilot: avaliação no Colab
Ative GPU e Internet antes de executar. Configure o Secret HF_TOKEN sem colocar seu valor nas células.
Execute em ordem em um kernel novo. Nenhum resultado real foi gerado na preparação local.
Use arquivos diferentes para Colab e Kaggle. Retomada exige o mesmo código, parâmetros e ambiente.

## 1. Uma GPU visível e verificação segura

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Antes de importar Torch ou inicializar CUDA.
import torch
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Ative um runtime GPU antes de continuar.')
print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.device_count() == 1, 'Reinicie o kernel para aplicar CUDA_VISIBLE_DEVICES.'

## 2. Segredos e caminhos
Autorize pessoalmente a montagem do Drive. O JSONL será gravado nele.

In [ ]:
from pathlib import Path
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive')
assert os.environ.get('HF_TOKEN'), 'Configure o Secret HF_TOKEN.'
os.environ['CHROMA_PATH'] = '/content/chroma'
os.environ['DATA_EVAL_DIR'] = 'data/eval/human-v2'
os.environ['EMBEDDING_DEVICE'] = 'cpu'  # Reserva VRAM para o Mistral.
output = Path('/content/drive/MyDrive/PartnerCopilot/generation-colab-v1.jsonl')
output.parent.mkdir(parents=True, exist_ok=True)

## 3. Clonar e instalar
O repositório é público. Não use token do GitHub na URL. Não faça pull automático ao retomar.

In [ ]:
import subprocess, sys
repo = Path('/content/partner-copilot')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/JoaoNordemann/partner-copilot.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## 4. Reconstruir o índice isolado
Usamos as 100 reviews do corpus human-v2, iguais às da avaliação local. Não ingerir o gold no índice de produção. Reexecutar esta célula cria outro diretório isolado.

In [ ]:
import json, uuid
from partner_copilot.config import Settings
from partner_copilot.evaluation.dataset import load_eval_set
from partner_copilot.ingestion.embedder import SentenceTransformerEmbedder
from partner_copilot.ingestion.chroma_store import ChromaStore
from partner_copilot.retrieval.retriever import Retriever
settings = Settings()
settings.export_hf_env()
questions, corpus, manifest = load_eval_set(settings.data_eval_dir)
store = ChromaStore(settings.chroma_path / ('eval-' + uuid.uuid4().hex),
    collection='isolated_generation_eval', embedder=SentenceTransformerEmbedder(settings), settings=settings)
store.upsert(corpus)
assert store.count() == len(corpus) == 100
retriever = Retriever(store, settings=settings)
print('Perguntas:', len(questions), 'Documentos:', store.count())

## 5. Paridade do retrieval antes de carregar o Mistral
Comparamos ranking sem corte e resultado operacional com o relatório local, mantendo os filtros. q13 pode retornar vazio por causa do limiar. A comparação de scores é registrada; diferenças de IDs interrompem a execução para investigação.

In [ ]:
report = json.loads(Path('reports/human-v2/retrieval-comparison.json').read_text(encoding='utf-8'))
assert report['snapshot_sha256_lf'] == manifest['file_sha256_lf'], 'Snapshot diferente.'
assert settings.embedding_model.rsplit('/', 1)[-1] == report['model'].rsplit('/', 1)[-1]
assert settings.retrieval_k == 8 and settings.retrieval_min_score == report['semantic_operational_min_score']
reference = {q['id']: q for q in report['methods']['semantic']['com_filtro']['questions']}
ranker = Retriever(store, settings=settings.model_copy(update={'retrieval_min_score': 0.0}))
parity = []
for identifier in ('q01', 'q13', 'q25'):
    q = next(q for q in questions if q.id == identifier)
    filters = q.filtros.model_dump(exclude_none=True)
    ranked = ranker.retrieve(q.pergunta, **filters)
    operational = retriever.retrieve(q.pergunta, **filters)
    expected = reference[identifier]
    row = {'id': identifier, 'ranked_ids': [d.review.id for d in ranked],
           'local_ranked_ids': expected['ranked_ids'], 'scores': [d.score for d in ranked],
           'local_scores': expected['scores'], 'operational_ids': [d.review.id for d in operational],
           'local_operational_ids': expected['operational_ids']}
    row['same_ids'] = row['ranked_ids'] == row['local_ranked_ids'] and row['operational_ids'] == row['local_operational_ids']
    parity.append(row)
output.with_suffix('.retrieval-check.json').write_text(json.dumps(parity, indent=2), encoding='utf-8')
print(json.dumps(parity, indent=2))
assert all(row['same_ids'] for row in parity), 'Investigue modelo, filtros, snapshot e empates antes de continuar.'

## 6. Carregar uma vez e testar q01
O teste é salvo separadamente, inclusive sua saída bruta. Falhas de contrato não são escondidas. A mesma instância do modelo será usada na rodada completa. Não altere o prompt após ver o teste sem registrar nova versão.

In [ ]:
import hashlib
from importlib.metadata import version
from partner_copilot.generation.provider import get_llm
from partner_copilot.evaluation.generation import RecordingRetriever, RecordingLLM, run_generation
from partner_copilot.pipeline import RAGPipeline
# Mesma identidade de ambiente usada pela CLI; nenhum segredo é incluído.
source_digest = hashlib.sha256()
for source in sorted((repo / 'src/partner_copilot').rglob('*.py')):
    source_digest.update(str(source.relative_to(repo)).replace('\\', '/').encode())
    source_digest.update(source.read_text(encoding='utf-8').encode())
source_digest.update((repo / 'scripts/evaluate_generation.py').read_text(encoding='utf-8').encode())
environment = {'python': sys.version.split()[0],
    'packages': {name: version(name) for name in ('torch', 'transformers', 'bitsandbytes', 'accelerate', 'chromadb', 'sentence-transformers')},
    'gpu_names': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    'torch_cuda_version': torch.version.cuda,
    'source_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'source_code_sha256': source_digest.hexdigest()}
output.with_suffix('.environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
llm = get_llm(settings)
q = next(q for q in questions if q.id == 'q01')
trace = {}
smoke = {'question_id': q.id, 'environment': environment}
try:
    answer = RAGPipeline(RecordingRetriever(retriever, trace, settings),
        RecordingLLM(lambda: llm, trace), settings=settings).ask(q.pergunta, **q.filtros.model_dump(exclude_none=True))
    smoke['answer'] = answer.model_dump()
except Exception as exc:
    smoke['error_type'] = type(exc).__name__
finally:
    smoke.update(trace)
    output.with_suffix('.smoke.json').write_text(json.dumps(smoke, ensure_ascii=False, indent=2), encoding='utf-8')
print(smoke.get('answer', {'error_type': smoke.get('error_type')}))
assert 'error_type' not in smoke, 'Leia o artefato smoke antes da rodada completa.'

## 7. Executar as 30 perguntas
Seed e temperatura vêm do Settings. O runner salva cada pergunta com flush/fsync, saída bruta, IDs, abstenção em answer, citações, latência e VRAM em generation_metrics. O ambiente está em cada linha, sem acrescentar linhas especiais ao JSONL. Retomada não repete erros já registrados.

In [ ]:
rows = run_generation(settings.data_eval_dir, output, settings, retriever,
    lambda: llm, environment=environment, backend='real_mistral_nf4', resume=True)
print('Tentativas salvas:', len(rows), 'Arquivo:', output)

## 8. Ambiente, métricas e revisão
Taxas finais exigem 30 tentativas. Leia pelo menos as dez respostas selecionadas, incluindo todas as negativas. IDs válidos não garantem apoio factual. A duração depende do runtime; não há benchmark real na preparação local.

In [ ]:
import runpy
helpers = runpy.run_path(str(repo / 'scripts/evaluate_generation.py'))
rows = helpers['read_results'](output)
summary = helpers['write_summary'](output, rows)
print(json.dumps(environment, indent=2))
print(json.dumps(summary, ensure_ascii=False, indent=2))
print(output.with_suffix('.review.md').read_text(encoding='utf-8'))
print('Preserve o JSONL e os arquivos environment, retrieval-check, smoke, summary e review.')